# Gold 1: patient_journey

One row per hospital admission. Config driven: schemas, the readmission window,
age buckets, the ICU unit definition and all Genie comments live in
gold_config.json, following the same volume pattern as bronze and silver.

In [0]:
# config home for the gold layer,
for sql in [
    "CREATE CATALOG IF NOT EXISTS cdr_gold",
    "CREATE SCHEMA  IF NOT EXISTS cdr_gold.config",
    "CREATE SCHEMA  IF NOT EXISTS cdr_gold.mimic_iv",
    "CREATE VOLUME  IF NOT EXISTS cdr_gold.config.gold_config",
]:
    spark.sql(sql); print("ok:", sql)

gold_config = r"""{
  "sources": {
    "silver_schema": "cdr_silver.processed_mimic_iv",
    "bronze_schema": "cdr_bronze.raw_mimic_iv"
  },
  "target": {
    "gold_schema": "cdr_gold.mimic_iv"
  },
  "parameters": {
    "include_ed_in_occupancy": false,
    "readmission_window_days": 30,
    "icu_careunit_patterns": ["intensive care", "(ccu)"],
    "urine_label_regex": "urine|foley",
    "ventilation_label_regex": "ventilat|intubat",
    "age_group_bounds": [30, 50, 65, 80]
  },
  "tables": {
    "patient_journey": {
      "comment": "One row per hospital admission with demographics, outcomes, risk scores and ICU exposure. Dates are de-identified and shifted per patient.",
      "column_comments": {
        "age_at_admission": "Derived: anchor_age + (admission year - anchor_year). MIMIC-IV stores no direct age.",
        "readmit_30d": "True if the patient was readmitted within the configured window after discharge. Null for the last known admission.",
        "ed_boarding_hours": "Hours between ED registration and ED exit, where recorded.",
        "drg_severity": "APR-DRG severity score (1-4), higher is more severe.",
        "icu_hours": "Total hours in ICU-type units for this admission, derived from transfers."
      }
    },
    "icu_clinical_monitoring": {
      "comment": "One row per ICU stay per day: fluids, urine output, weight, procedures and ventilation. Fluid intake comes from ingredientevents (ml only).",
      "column_comments": {
        "intake_ml": "Daily fluid intake in ml from ingredientevents. Raw inputevents mixes units and is not summed here.",
        "net_balance_ml": "intake_ml minus output_ml for the day. Negative means the patient lost more than they received.",
        "icu_day_number": "Day of stay starting at 1, counted from the first recorded event of the stay."
      }
    },
    "hospital_pressure_daily": {
      "comment": "Daily occupancy, admissions and transfer activity per care unit. Dates are per-patient shifted: compare day_of_week and is_weekend across patients, not absolute dates.",
      "column_comments": {
        "patients_present": "Distinct patients physically in the unit that day, from interval-expanded transfer segments.",
        "pressure_score": "Project-defined z-score blend of occupancy, admissions and transfers per unit. Not a bed-capacity percentage.",
        "day_of_week": "Preserved by the de-identification shift, safe to compare across patients.",
        "is_weekend": "True for Saturday and Sunday. Preserved by the de-identification shift."
      }
    }
  }
}"""

dbutils.fs.put("/Volumes/cdr_gold/config/gold_config/gold_config.json",
               gold_config.strip(), overwrite=True)
import json
print("config ok, tables:", list(json.loads(gold_config)["tables"].keys()))

In [0]:
# gold config loader: sources, tunable parameters and Genie comments come from the
# JSON written by the setup cell above, same volume pattern as bronze and silver.
import json
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

with open("/Volumes/cdr_gold/config/gold_config/gold_config.json", "r") as f:
    GCFG = json.load(f)

SILVER = GCFG["sources"]["silver_schema"]
BRONZE = GCFG["sources"]["bronze_schema"]
GOLD   = GCFG["target"]["gold_schema"]
PARAMS = GCFG["parameters"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

def t(name):
    # prefer silver; fall back to bronze so this still runs while some silver tables are pending
    for schema in (SILVER, BRONZE):
        try:
            df = spark.table(f"{schema}.{name}")
            if schema == BRONZE:
                print(f"note: using BRONZE for {name} (silver not found)")
            return df
        except Exception:
            continue
    raise Exception(f"table {name} not found in {SILVER} or {BRONZE}")

def t_any(*names):
    # try several table names (the team silver renames some tables)
    for n in names:
        try:
            return t(n)
        except Exception:
            continue
    raise Exception(f"none of {names} found")

def title_col(df):
    return "description" if "description" in df.columns else "long_title"

def apply_comments(table_key):
    # table and column comments from the config
    meta = GCFG["tables"][table_key]
    spark.sql(f"COMMENT ON TABLE {GOLD}.{table_key} IS '{meta['comment']}'")
    for col_name, comment in meta.get("column_comments", {}).items():
        spark.sql(f"COMMENT ON COLUMN {GOLD}.{table_key}.{col_name} IS '{comment}'")
    print(f"comments applied to {table_key}")

def age_group_expr(colname):
    # age buckets built from the configured bounds
    bounds = PARAMS["age_group_bounds"]
    expr, lo = None, 18
    for b in bounds:
        label = f"{lo}-{b-1}"
        expr = F.when(F.col(colname) < b, label) if expr is None else expr.when(F.col(colname) < b, label)
        lo = b
    return expr.otherwise(f"{bounds[-1]}+")

GOLD_AUDIT = f"{GOLD}.gold_audit_log"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_AUDIT} (
    run_id           STRING,
    target_table     STRING,
    status           STRING,
    row_count        BIGINT,
    start_time       TIMESTAMP,
    end_time         TIMESTAMP,
    duration_seconds DOUBLE,
    notes            STRING
) USING DELTA
""")

RUN_ID = f"gold_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"
_build_start = {}

def audit_start(table_key):
    _build_start[table_key] = datetime.now()

def audit_end(table_key, row_count, notes=None):
    end = datetime.now()
    start = _build_start.get(table_key, end)
    dur = (end - start).total_seconds()
    spark.createDataFrame(
        [(RUN_ID, f"{GOLD}.{table_key}", "SUCCESS", row_count, start, end, dur, notes)],
        "run_id string, target_table string, status string, row_count long, "
        "start_time timestamp, end_time timestamp, duration_seconds double, notes string"
    ).write.mode("append").format("delta").saveAsTable(GOLD_AUDIT)
    print(f"audit: {table_key} | {row_count:,} rows | {dur:.1f}s | run {RUN_ID}")

print(f"gold loader ready | schema: {GOLD} | run: {RUN_ID}")

In [0]:
# base admission, age, and readmission
adm = t("admissions")
pat = t("patients")

# derived the age: MIMIC-IV only stores anchor values
base = (adm.join(pat.select("subject_id","anchor_age","anchor_year","gender","dod"), "subject_id", "left")
  .withColumn("age_at_admission", F.col("anchor_age") + (F.year("admittime") - F.col("anchor_year")))
  .withColumn("age_group", age_group_expr("age_at_admission"))
  .withColumn("los_days", F.round((F.unix_timestamp("dischtime") - F.unix_timestamp("admittime")) / 86400, 2))
  .withColumn("ed_boarding_hours",
      F.round((F.unix_timestamp("edouttime") - F.unix_timestamp("edregtime")) / 3600, 2))
)

# readmission: intervals within one patient survive the date shift,
# the window length comes from the config (readmission_window_days)
w = Window.partitionBy("subject_id").orderBy("admittime")
base = (base
  .withColumn("next_admittime", F.lead("admittime").over(w))
  .withColumn("days_to_next_admission", F.datediff(F.col("next_admittime"), F.col("dischtime")))
  .withColumn("readmit_30d",
      F.when(F.col("days_to_next_admission").between(0, PARAMS["readmission_window_days"]), True)
       .when(F.col("days_to_next_admission").isNull(), None)
       .otherwise(False))
  .drop("next_admittime"))

In [0]:
# diagnosis, procedure, drg and icu from the transfers
# primary diagnosis and procedure: seq_num = 1, composite key (icd_code, icd_version).
# the team silver renames some tables (diagnoses_icd -> icd_diagnoses) and the
# dictionary column (long_title -> description), so both names are handled.
di = t_any("icd_diagnoses", "diagnoses_icd")
dd = t("d_icd_diagnoses")
pi = t_any("icd_procedures", "procedures_icd")
dp = t("d_icd_procedures")

prim_dx = (di.filter("seq_num = 1")
  .join(dd.select("icd_code","icd_version", F.col(title_col(dd)).alias("primary_diagnosis")),
        ["icd_code","icd_version"], "left")
  .select("hadm_id", "primary_diagnosis"))

prim_proc = (pi.filter("seq_num = 1")
  .join(dp.select("icd_code","icd_version", F.col(title_col(dp)).alias("primary_procedure")),
        ["icd_code","icd_version"], "left")
  .select("hadm_id", "primary_procedure"))

proc_count = pi.groupBy("hadm_id").agg(F.count("*").alias("procedure_count"))
dx_count   = di.groupBy("hadm_id").agg(F.count("*").alias("diagnosis_count"))

# DRG severity and mortality: isk scores, APR rows only
drg = t("drgcodes")
drg_risk = (drg.filter(F.upper("drg_type").contains("APR"))
  .groupBy("hadm_id")
  .agg(F.max("drg_severity").alias("drg_severity"),
       F.max("drg_mortality").alias("drg_mortality")))

# ICU metrics from transfers. The unit patterns come from the config, so may
# change the ICU definition if want to. eventtype filter: discharge
# rows have no real careunit (for the 'UNKNOWN').
tr = t("transfers")
icu_cond = None
for p in PARAMS["icu_careunit_patterns"]:
    c = F.lower("careunit").contains(p)
    icu_cond = c if icu_cond is None else (icu_cond | c)

icu_rows = (tr
  .filter(F.col("eventtype").isin("admit", "transfer"))
  .filter(icu_cond))

icu_metrics = (icu_rows.groupBy("hadm_id")
  .agg(F.count("*").alias("icu_transfer_segments"),
       F.round(F.sum((F.unix_timestamp("outtime") - F.unix_timestamp("intime")) / 3600), 1).alias("icu_hours")))

In [0]:
# assemble, save, and audit
audit_start("patient_journey")

gold1 = (base
  .join(prim_dx, "hadm_id", "left")
  .join(prim_proc, "hadm_id", "left")
  .join(proc_count, "hadm_id", "left")
  .join(dx_count, "hadm_id", "left")
  .join(drg_risk, "hadm_id", "left")
  .join(icu_metrics, "hadm_id", "left")
  .withColumn("had_icu_stay", F.col("icu_hours").isNotNull())
  .select("subject_id","hadm_id","admittime","dischtime","los_days",
          "age_at_admission","age_group","gender",
          "insurance","language","marital_status","race",
          "admission_type","admission_location","discharge_location",
          "ed_boarding_hours","hospital_expire_flag",
          "days_to_next_admission","readmit_30d",
          "primary_diagnosis","primary_procedure","diagnosis_count","procedure_count",
          "drg_severity","drg_mortality",
          "had_icu_stay","icu_transfer_segments","icu_hours"))

gold1.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD}.patient_journey")
n = spark.table(f"{GOLD}.patient_journey").count()
print(f"{GOLD}.patient_journey: {n:,} rows")

audit_end("patient_journey", n, notes="built from admissions, patients, icd, drg, transfers")

# table and column comments come from the config
apply_comments("patient_journey")

display(spark.table(f"{GOLD}.patient_journey").limit(5))

In [0]:
gj  = spark.table(f"{GOLD}.patient_journey")
icm = spark.table(f"{GOLD}.icu_clinical_monitoring")

print("===outlier: patient_journey ===")
print("LOS negatif:", gj.filter("los_days < 0").count())
print("ED boarding negative:", gj.filter("ed_boarding_hours < 0").count())
print("LOS > 100 days:", gj.filter("los_days > 100").count())
print("ED boarding > 48 hrs:", gj.filter("ed_boarding_hours > 48").count())
display(gj.orderBy(F.desc("los_days"))
          .select("hadm_id","los_days","age_at_admission","admission_type","discharge_location")
          .limit(5))

print("=== outlier: icu_clinical_monitoring ===")
print("intake > 20 L/day:", icm.filter("intake_ml > 20000").count())
print("output > 15 L/day:", icm.filter("output_ml > 15000").count())
display(icm.orderBy(F.abs("net_balance_ml").desc())
           .select("stay_id","clinical_date","intake_ml","output_ml","net_balance_ml","icu_day_number")
           .limit(5))

In [0]:
print("icu_hours > 2000 hrs (>83 days ICU):", gj.filter("icu_hours > 2000").count())
display(gj.orderBy(F.desc("icu_hours")).select("hadm_id","icu_hours","los_days","age_group").limit(5))